In [ ]:
# Example 1: parsers.CellArgs + FixArgs round-trip through pydantic.
from jupyter_ai_magic_commands.parsers import CellArgs, FixArgs, FORMAT_CHOICES

cell = CellArgs(model_id="openai:gpt-4o-mini", format="markdown")
assert cell.type == "root"
assert cell.model_id == "openai:gpt-4o-mini"
assert cell.format == "markdown"
assert cell.region_name is None

fix = FixArgs(model_id="anthropic:claude-3-haiku", format="text", api_base="https://example.invalid")
assert fix.type == "fix"
assert fix.api_base == "https://example.invalid"

# Round-trip through JSON
round_tripped = CellArgs.model_validate_json(cell.model_dump_json())
assert round_tripped == cell

assert "markdown" in FORMAT_CHOICES and "json" in FORMAT_CHOICES
print("FORMAT_CHOICES:", FORMAT_CHOICES)
print("cell:", cell)
print("fix:", fix)


In [ ]:
# Example 2: alias-management arg models (RegisterArgs / UpdateArgs / DeleteArgs / ResetArgs / ListArgs / VersionArgs / HelpArgs).
from jupyter_ai_magic_commands.parsers import (
    DeleteArgs,
    HelpArgs,
    ListArgs,
    RegisterArgs,
    ResetArgs,
    UpdateArgs,
    VersionArgs,
)

reg = RegisterArgs(name="gpt4", target="openai:gpt-4o", api_base="https://api.openai.com")
upd = UpdateArgs(name="gpt4", target="openai:gpt-4o-mini")
deal = DeleteArgs(name="gpt4")
lst = ListArgs(provider_id="openai")

assert reg.type == "alias"
assert upd.type == "update"
assert deal.type == "dealias"
assert lst.type == "list" and lst.provider_id == "openai"
assert ResetArgs().type == "reset"
assert VersionArgs().type == "version"
assert HelpArgs().type == "help"
print("register:", reg)
print("update:", upd)
print("list:", lst)


In [ ]:
# Example 3: models.persona — Persona model + bundled JupyternautPersona.
from jupyter_ai_magic_commands.models.persona import (
    JUPYTERNAUT_AVATAR_ROUTE,
    JupyternautPersona,
    Persona,
)

assert isinstance(JupyternautPersona, Persona)
assert JupyternautPersona.name == "Jupyternaut"
assert JupyternautPersona.avatar_route == JUPYTERNAUT_AVATAR_ROUTE

custom = Persona(name="Pixie", avatar_route="api/static/pixie.svg")
assert custom.name == "Pixie"

# Validation: missing required field raises
from pydantic import ValidationError
try:
    Persona(name="NoAvatar")
    raise AssertionError("expected ValidationError")
except ValidationError:
    pass

print("jupyternaut:", JupyternautPersona)
print("custom:", custom)


In [ ]:
# Example 4: models.completion — inline-completion request / item / reply / stream chunk models.
from jupyter_ai_magic_commands.models.completion import (
    CompletionError,
    InlineCompletionItem,
    InlineCompletionList,
    InlineCompletionReply,
    InlineCompletionRequest,
    InlineCompletionStreamChunk,
)

req = InlineCompletionRequest(
    number=1, prefix="def add(a, b):\n    return ", suffix="", mime="text/x-python", stream=False
)
assert req.number == 1 and req.path is None

item = InlineCompletionItem(insertText="a + b")
reply = InlineCompletionReply(list=InlineCompletionList(items=[item]), reply_to=1)
assert reply.reply_to == 1 and reply.list.items[0].insertText == "a + b"

chunk = InlineCompletionStreamChunk(response=item, reply_to=1, done=True)
assert chunk.type == "stream" and chunk.done is True

err = CompletionError(type="ImportError", title="oops", traceback="...")
assert err.type == "ImportError"
print("reply:", reply.model_dump())


In [ ]:
# Example 5: top-level ipython-extension entry points are exposed + invoking the click cell-magic parser through standalone_mode=False.
import jupyter_ai_magic_commands
from jupyter_ai_magic_commands.parsers import CellArgs, cell_magic_parser, line_magic_parser

# Public extension entry points exist (per IPython %load_ext contract)
assert callable(jupyter_ai_magic_commands.load_ipython_extension)
assert callable(jupyter_ai_magic_commands.unload_ipython_extension)
assert jupyter_ai_magic_commands.__version__

# click command objects are exposed
import click
assert isinstance(cell_magic_parser, click.Command)
assert isinstance(line_magic_parser, click.Group)

# Invoke the cell-magic parser without click exiting the process; it returns a CellArgs.
parsed = cell_magic_parser.main(
    args=["openai:gpt-4o-mini", "--format", "json"],
    standalone_mode=False,
)
assert isinstance(parsed, CellArgs)
assert parsed.model_id == "openai:gpt-4o-mini"
assert parsed.format == "json"
print("parsed:", parsed)
